_ExplainDB — Copyright (C) 2026 Prof. Dr. Jens Dittrich, Saarland University._  
_Licensed under the GNU Affero General Public License v3 (AGPL-3.0). See the `LICENSE` file._

# Replacement Selection Playground

External merge sort first makes sorted **runs**, then merges them. Sorting chunks in memory reads M elements, sorts them
and writes them out: every run has M elements (the last one maybe fewer). **Replacement selection** makes runs that are
about twice as long on random input, and a single run on sorted input, so that fewer runs are left to merge, which can
save a whole merge pass.

Main memory is one array of M places. It holds a **heap** at its front and a **list** behind it, together never more
than M elements: |heap| + |list| ≤ M. The heap holds the elements of the current run, the list those that wait for the
next run. Each time the heap gives up its last place, the border between the two moves one place to the front; while
the input lasts, the list grows towards the front, its newest element first, right behind the heap.

The heap is kept in **half order** (textbooks call this the heap property): no child is smaller than its parent. So no
element is smaller than the root, but the heap is not sorted (the two children of a node, for example, can be in either
order). The heap's places form a tree: place 0 is the root, and the children of place i are at places 2i + 1 and
2i + 2. The playground draws this front part of memory as a tree.

Each step works as follows:

- The root of the heap, its smallest element, is written to the current output run; it is now the **last element on
  output**.
- The next element is drawn from the input R and compared with the last element on output. If the new element is not
  smaller (green), it can still be written to the current run: it takes the root's place in the heap. If it is smaller
  (red), it has to wait for the next run: the heap gives up its last place, the element there takes the root's place,
  and the new element goes into the freed place, as the list's newest element.
- When the input is empty, nothing is drawn: the heap gives up its last place, which stays empty, and the element there
  takes the root's place.
- In all cases, the element now at the root is sifted down (unless it is the heap's only element): its two children are
  compared with each other, then the element is compared with the smaller child. If the child is smaller, the two swap.
  This repeats one level lower, until no child is smaller or the element has reached a leaf.
- When the heap is empty, the current run is complete: the list is **bulk-loaded** into a new heap, in place, and the
  next run starts. If the input ended while the heap still held elements, the outputs after that left empty places in
  front of the list; the list then first moves to the front of memory, in its order, closing them.

The playground follows `ReplacementSelection` in `system/query_processing/sorting.py` step by step. It starts with the
example of the lecture's slides: M = 4, the input 30 20 10 40 25 73 16 26 33 50 31, and the runs 10 20 25 30 40 73 and
16 26 31 33 50. Everything it shows comes from that code: the playground runs the code and checks every step it shows
against it.

- **preset:** the slides' example, or **n** distinct values (1 to 60) drawn at random; the **seed** decides which
  values. They are shown as drawn (**random**), **sorted**, **reversed**, or **almost sorted** (sorted, with about every
  tenth pair of neighbours swapped). n and seed are greyed out for the slides' example and for your own input.
- **input R:** the input, whole numbers from 0 to 999 separated by spaces or commas, at most 60 of them, so that
  everything fits on the screen; values may repeat. Type your own and press Enter: the preset shows **own input**.
- **M:** how many places main memory has, which heap and list share, from 3 to 12. A change of the preset, n, the seed,
  the input or M starts again at the first step.
- **Heap maintenance step by step:** off (the default) or on. When a bulk load starts, the list's places become the
  heap's, in place, and the tree shows them in array order (place 0 is the root, places 1 and 2 are below it, and so
  on); this tree need not be a heap yet.
  - **Off:** a bulk load shows only its result, the heap in half order, with the places whose element changed marked
    grey. After an output, the step in which an element takes the root's place already shows it sifted down, again with
    the changed places marked.
  - **On:** the bulk load shows how the tree becomes a heap. First it shows which places are leaves, with nothing to
    sift. Then it sifts down every inner node (dashed), from the last one back to the root; each comparison and each
    swap is a step of its own, and the heap in half order ends the bulk load. After an output, the element at the root
    is sifted down the same way.
  - Switching keeps your place.
- **◀ back** and **forward ▶** go one step; **⏮︎ start** and **end ⏭︎** jump to the first and the last step; **next
  output ⏩︎** jumps to the next element written to the output, past the draws, comparisons, swaps and bulk loads in
  between. The caption says what each step does, in the words of the slides.
- **The view:** the input R, where every element already taken leaves an empty cell; below it the comparison box, as on
  the slides: the last element on output, the next element of the input and the decision, "25 >= 10 → heap (this run)"
  on green or "16 >= 25 → list (next run)" on red. The box always writes the test with >=, as the slides do: green means
  it holds, red means it does not. The decision appears in the step that writes the root to the output and stays until
  the next output or bulk load. Below the box: the heap as a tree, its root on top framed amber, the children of every
  node below it, and left of every node its place in memory. Below the tree: main memory as one row of M cells, each
  with its place below it: the heap's places, a bar, then the list's places; an empty place is dashed. A new element
  that goes to the list waits in the comparison box until the heap has given up its place and the root has sunk. Then
  the output runs, each in a box (numbered from run 0). Every new element enters the output on the left and pushes the
  others to the right, so a complete run reads from its largest element on the left to its smallest on the right, and
  the newest run stands leftmost. The colours tell the runs apart (three colours, which repeat after three runs): the
  heap holds elements of the current run, the list those of the next run. Grey marks what a step changes and the
  elements of the input it reads; with **Heap maintenance step by step** on, blue marks the two elements a sift
  compares: first the two children of the sinking element, then the sinking element and its smaller child (these two
  are also joined by a thick blue edge).

In [ ]:
import sys, pathlib

# Put the repo root (the directory containing `system/`) on sys.path so
# `from system...` works whether this notebook is opened from its own
# folder or from the repo root.
for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_p / "system").is_dir():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break

In [ ]:
from system.playground.replacement_selection_widgets import (
    ReplacementSelectionPlayground,
)

ReplacementSelectionPlayground().widget

## Things to try

- **The slides' example, step by step:** press forward ▶ and compare with the slides. 25 goes into the heap (green, 25 ≥
  10), 16 goes to the list (red, 16 < 25): the heap gives up place 3, and 16 goes there once 73 has sunk. Watch the bar
  in the row of memory move to the front as the list grows. After 73 the heap is empty and memory holds the list
  50 33 26 16, newest first; the bulk load makes it the heap 16 33 26 50 of run 1. The slides draw the heap as a sorted
  sequence; here you see the real heap, as an array drawn as a tree. Then switch **Heap maintenance step by step** on
  and go through it again, with every comparison and swap.
- **Sorted input:** choose **sorted**. Every drawn element is at least as large as the last output, so all of them go
  into the heap: a single run, however long the input. Sorting chunks of M would make n / M runs (rounded up).
- **Reversed input:** choose **reversed**. Every drawn element is smaller than the last output, so all of them go to the
  list: runs of M elements (the last one maybe fewer), no better than sorting chunks.
- **Random input:** choose **random**, set M to 3 or 4 and n to 60, and press end ⏭︎. The last caption gives the run
  lengths and their average: most runs are about 2 × M long. The average stays a little below (about 1.8 × M), because
  the last run is cut short when the input ends. Try a few seeds. With a larger M there are only a few runs, so the
  short last run pulls the average down further.
- **Almost sorted input:** choose **almost sorted**. How long are the runs now?
- **Equal values:** type `5 5 3 5 1 5` with M = 3. An element equal to the last output still goes into the heap, as it
  may be written to the current run; an element sifting down stops above an equal child, and of two equal children the
  sift takes the left one.
- **The list moves to the front:** type `30 20 10 40 25 73 16 26`, the slides' input without its last three values,
  with M = 4. The input ends while the heap still holds 40 and 73, with the list 26 16 behind it. Writing them leaves
  places 1 and 0 empty, in front of the list. Before run 1 the list moves to the front of memory, in its order, closing
  the empty places, and is bulk-loaded there.
- **The bulk load:** type `70 60 50 40 30 20 10`, set M to 7 and switch **Heap maintenance step by step** on. The bulk
  load sifts down the inner nodes (dashed) from the last one back to the root; on every level it compares the two
  children, then the sinking element with the smaller one (blue). The root 70 sinks two levels. It works in place: each
  swap in the tree swaps two cells in the row of memory below it.

## Why the bulk load takes linear time

Inserting n elements into a heap one by one costs up to log n swaps each, about n log n in all. The bulk load (Floyd's
method, `bulkload` in `sorting.py`) instead puts the elements into the array as they come and sifts down every inner
node, from the last one back to the root. Half of the nodes are leaves and are not sifted at all; a quarter lie one
level above them and sift down at most one level, an eighth at most two levels, and so on. Summed up, that is at most n
swaps and at most 2n comparisons (two per level an element can sink), however large n is. The playground bulk-loads at
most M ≤ 12 elements at a time, so both ways take only a few steps there; the difference shows for large heaps.

Sources: the bulk load is from Robert W. Floyd, [Algorithm 245: Treesort 3](https://doi.org/10.1145/355588.365103),
Communications of the ACM 7(12):701, December 1964. The heap itself, built by inserting the elements one by one, is from
J. W. J. Williams, Algorithm 232: Heapsort, Communications of the ACM 7(6):347–348, June 1964.

## What the model assumes

- Elements, not pages: memory is counted in elements, as on the slides ("we will ignore pages for a moment"). Real
  systems read and write pages, and records may differ in size.
- Memory counts the heap and the list only, not the buffers that read the input and write the output, as on the slides.
- The first M elements go into the list, as in the code, and the first bulk load turns them into the heap of run 0, the
  same way as the list of every later run. The slides load them into the heap directly.
- The slides draw the heap and the list as two separate areas. Here, as in the code and in textbooks, they share one
  array of M places: each element that waits for the next run takes the place the heap has just given up, so the list
  holds its elements newest first. A run's heap is therefore built from the list in another order than on the slides:
  in the slides' example, run 1's heap is 16 33 26 50, not 16 26 33 50. Equal values may leave a run in another order
  too; the runs themselves hold the same elements.
- The heap is the textbook heap of `sorting.py` (`sift_down`, `bulkload`), so that every comparison and swap can be
  followed; an element sifting down stops above an equal child. Python's own heap module `heapq` sifts in a different
  way.